# Descargar las estaciones reales de Costa Rica que faltan (ECO | Wind)

Pablo notó que la muestra de 4 sitios precacheados (San José, Nicoya, Liberia, Finca Favorita) es
pobre para un territorio tan accidentado como Costa Rica -- Hallazgo 21/22/23 ya mostraron que la
forma real del viento cambia mucho de una zona a otra. El catálogo completo
(`datos_clima/epw_catalog_global.json`, 5,276 estaciones/20 países, Hallazgo 19) en realidad sólo
tiene **12 estaciones para Costa Rica en total** -- ya tenemos 4, faltan **8**. Este notebook las
descarga todas, igual que ya lo hace la app (`descargar_y_extraer_epw()`, mismo patrón que
DDP-lite/Skyplus) -- no hay que ir sitio por sitio a mano.

**No corre en el sandbox de desarrollo** (climate.onebuilding.org bloqueado, Hallazgo 2) -- correr
esto en Colab. Al final arma un .zip con los 8 EPW nuevos para descargar y subir de vuelta al chat
(mismo mecanismo que ya se usó para los primeros 3 EPW reales, Hallazgo 18).

In [1]:
import os

def _find_repo_root():
    for candidato in ("..", "/content/ECO-Wind"):
        if os.path.exists(os.path.join(candidato, ".git")):
            return os.path.abspath(candidato)
    return None

repo = _find_repo_root()
if repo is None:
    repo = "/content/ECO-Wind"
    get_ipython().system(f"git clone https://github.com/Sogo2012/ECO-Wind.git {repo}")
else:
    get_ipython().system(f"git -C {repo} fetch origin main")
    get_ipython().system(f"git -C {repo} reset --hard origin/main")

get_ipython().run_line_magic("cd", f"{repo}/notebooks")
get_ipython().system(f"git -C {repo} log -1 --format='Commit activo: %h  %s  (%ci)'")

From https://github.com/Sogo2012/eco-wind
 * branch            main       -> FETCH_HEAD


HEAD is now at 8d226b6 docs(fase2): Hallazgo 28 -- ERA5 real, mejor que NASA POWER pero no le gana a GWA


/home/user/eco-wind/notebooks
Commit activo: 8d226b6  docs(fase2): Hallazgo 28 -- ERA5 real, mejor que NASA POWER pero no le gana a GWA  (2026-09-01 00:38:30 +0000)


In [2]:
import sys
sys.path.insert(0, "..")

import json
import shutil
import zipfile

from engine.epw_real import (
    CARPETA_EPW_REAL, _SITIOS_PRECACHEADOS_COORDS, cargar_epw_real,
    descargar_y_extraer_epw, _haversine_km,
)

catalogo = json.load(open(os.path.join(repo, "datos_clima", "epw_catalog_global.json")))
cri = catalogo["CRI"]
print(f"Catálogo de Costa Rica: {len(cri)} estaciones en total.")

# Las 4 que ya tenemos, identificadas por nombre (son las únicas 4 de las 12 que ya están
# precacheadas -- ver _SITIOS_PRECACHEADOS_COORDS en engine/epw_real.py).
YA_TENEMOS = {"San Jose Santamaria Intl AP", "Nicoya AP", "Quiros Liberia Intl AP", "Finca Favorita"}
faltantes = [s for s in cri if s["name"] not in YA_TENEMOS]

print(f"Ya tenemos {len(YA_TENEMOS)} localmente. Faltan {len(faltantes)}:")
for s in faltantes:
    print(f"  - {s['name']} ({s.get('state', '?')})")

Catálogo de Costa Rica: 12 estaciones en total.
Ya tenemos 4 localmente. Faltan 8:
  - Limon Intl AP (LI)
  - Chacarita Puntarenas AP (PU)
  - Palmar Sur Southern Zone Intl AP (PU)
  - Parrita (PU)
  - Paso Canoas AP (PU)
  - Puntarenas (PU)
  - San Jose Bolanos Intl AP (SJ)
  - San Jose La Sabana (SJ)


## Descargar y verificar cada una

Para cada estación faltante: descarga el ZIP real de climate.onebuilding.org, extrae el .epw, lo
copia a `datos_clima/epw_real/` (misma carpeta que las otras 3), y lo vuelve a abrir con
`cargar_epw_real()` para confirmar que se lee bien -- reportando la media real de viento, la
elevación y la coordenada real que trae el propio encabezado del archivo (más confiable que la del
catálogo: la de Finca Favorita, por ejemplo, ya se sabía que no coincidía -- ver el comentario
arriba de `_SITIOS_PRECACHEADOS_COORDS` en `engine/epw_real.py`).

In [3]:
os.makedirs(CARPETA_EPW_REAL, exist_ok=True)
descargadas = []

for s in faltantes:
    print(f"=== {s['name']} ===")
    try:
        ruta_tmp = descargar_y_extraer_epw(s["url"])
        ruta_final = os.path.join(CARPETA_EPW_REAL, os.path.basename(ruta_tmp))
        shutil.copy(ruta_tmp, ruta_final)

        df, meta = cargar_epw_real(ruta_final)
        print(f"  OK -- {os.path.basename(ruta_final)}")
        print(f"  Media real: {df['WS10M'].mean():.3f} m/s | elevación: {meta['elevacion_m']:.0f} m | "
              f"lat={meta['lat']:.4f}, lon={meta['lon']:.4f}")

        # Distancia al sitio precacheado real más cercano de los 4 que ya teníamos --
        # da una primera idea de si esto aporta cobertura nueva o es redundante.
        dist_min = min(_haversine_km(meta["lat"], meta["lon"], slat, slon)
                        for slat, slon in _SITIOS_PRECACHEADOS_COORDS.values())
        print(f"  Distancia al más cercano de los 4 ya conocidos: {dist_min:.1f} km")

        descargadas.append(dict(nombre=s["name"], archivo=os.path.basename(ruta_final),
                                 media_m_s=float(df["WS10M"].mean()), elevacion_m=meta["elevacion_m"],
                                 lat=meta["lat"], lon=meta["lon"], dist_km_mas_cercano=dist_min))
    except Exception as exc:
        print(f"  FALLO: {exc!r}")
    print()

print(f"Descargadas con éxito: {len(descargadas)}/{len(faltantes)}")

=== Limon Intl AP ===
  FALLO: ProxyError(MaxRetryError("HTTPSConnectionPool(host='climate.onebuilding.org', port=443): Max retries exceeded with url: /WMO_Region_4_North_and_Central_America/CRI_Costa_Rica/CRI_LI_Limon.Intl.AP.787670_TMYx.2009-2023.zip (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))

=== Chacarita Puntarenas AP ===


  FALLO: ProxyError(MaxRetryError("HTTPSConnectionPool(host='climate.onebuilding.org', port=443): Max retries exceeded with url: /WMO_Region_4_North_and_Central_America/CRI_Costa_Rica/CRI_PU_Chacarita-Puntarenas.AP.787613_TMYx.2009-2023.zip (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))

=== Palmar Sur Southern Zone Intl AP ===


  FALLO: ProxyError(MaxRetryError("HTTPSConnectionPool(host='climate.onebuilding.org', port=443): Max retries exceeded with url: /WMO_Region_4_North_and_Central_America/CRI_Costa_Rica/CRI_PU_Palmar.Sur-Southern.Zone.Intl.AP.787720_TMYx.2009-2023.zip (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))

=== Parrita ===


  FALLO: ProxyError(MaxRetryError("HTTPSConnectionPool(host='climate.onebuilding.org', port=443): Max retries exceeded with url: /WMO_Region_4_North_and_Central_America/CRI_Costa_Rica/CRI_PU_Parrita.749036_TMYx.2009-2023.zip (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))

=== Paso Canoas AP ===


  FALLO: ProxyError(MaxRetryError("HTTPSConnectionPool(host='climate.onebuilding.org', port=443): Max retries exceeded with url: /WMO_Region_4_North_and_Central_America/CRI_Costa_Rica/CRI_PU_Paso.Canoas.AP.787606_TMYx.2009-2023.zip (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))

=== Puntarenas ===


  FALLO: ProxyError(MaxRetryError("HTTPSConnectionPool(host='climate.onebuilding.org', port=443): Max retries exceeded with url: /WMO_Region_4_North_and_Central_America/CRI_Costa_Rica/CRI_PU_Puntarenas.787600_TMYx.2009-2023.zip (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))

=== San Jose Bolanos Intl AP ===


  FALLO: ProxyError(MaxRetryError("HTTPSConnectionPool(host='climate.onebuilding.org', port=443): Max retries exceeded with url: /WMO_Region_4_North_and_Central_America/CRI_Costa_Rica/CRI_SJ_San.Jose-Bolanos.Intl.AP.787640_TMYx.2009-2023.zip (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))

=== San Jose La Sabana ===


  FALLO: ProxyError(MaxRetryError("HTTPSConnectionPool(host='climate.onebuilding.org', port=443): Max retries exceeded with url: /WMO_Region_4_North_and_Central_America/CRI_Costa_Rica/CRI_SJ_San.Jose-La.Sabana.787605_TMYx.2009-2023.zip (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))

Descargadas con éxito: 0/8


## Empaquetar para subir de vuelta al chat

Un solo .zip con los EPW nuevos + un resumen en JSON -- descargalo del navegador de archivos de
Colab (ícono de carpeta a la izquierda) y subilo acá en el chat. Con eso integro los sitios nuevos
en `engine/epw_real.py`, corro de nuevo Hallazgo 21-23 con la muestra ampliada, documento y hago
commit+push -- igual que con los primeros 3 EPW (Hallazgo 18).

In [4]:
ruta_zip = os.path.join(repo, "estaciones_cr_nuevas.zip")
with zipfile.ZipFile(ruta_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for d in descargadas:
        z.write(os.path.join(CARPETA_EPW_REAL, d["archivo"]), arcname=d["archivo"])
    z.writestr("resumen.json", json.dumps(descargadas, indent=2, ensure_ascii=False))

print(f"Armado: {ruta_zip} ({os.path.getsize(ruta_zip) / 1024:.0f} KB, {len(descargadas)} estaciones)")

try:
    from google.colab import files
    files.download(ruta_zip)
except ImportError:
    print("No estás en Colab -- el archivo quedó en el disco local, en la ruta de arriba.")

Armado: /home/user/eco-wind/estaciones_cr_nuevas.zip (0 KB, 0 estaciones)
No estás en Colab -- el archivo quedó en el disco local, en la ruta de arriba.


## Enfoque en Limón -- ¿mejor donante real para Finca Favorita?

Pablo pidió arrancar por acá primero: de las 8 estaciones nuevas, Limón es la más relevante para un
problema abierto real -- Finca Favorita falla catastróficamente con los 3 métodos de ajuste de
magnitud probados (NASA POWER, GWA, ERA5, Hallazgo 25/26/28), y ya se sabía (Hallazgo 22) que San
José -- su único donante disponible hoy, a 178.6km -- es el mejor de los 4 sitios conocidos tanto
por distancia como por forma. La pregunta: ¿Limón, en el mismo Caribe y mucho más cerca, es
genuinamente mejor?

Este análisis se corrió localmente (no necesita red -- son EPW ya descargados + funciones ya
existentes), así que los resultados de abajo son reales, no placeholders de sandbox.

In [5]:
from engine.epw_real import cargar_epw_real, heatmap_json_desde_epw, _haversine_km, CARPETA_EPW_REAL
from engine.formas_regionales import (
    cargar_formas_conocidas, excedencia_json_desde_epw_residual, vecino_mas_cercano, generar_clima_gwa,
)
from engine.simulador_pista_a import simular

ruta_limon = os.path.join(CARPETA_EPW_REAL, "CRI_LI_Limon.Intl.AP.787670_TMYx.2009-2023.epw")
df_limon, meta_limon = cargar_epw_real(ruta_limon)
print(f"Limón: media={df_limon['WS10M'].mean():.3f} m/s, elev={meta_limon['elevacion_m']:.1f}m, "
      f"lat={meta_limon['lat']:.4f}, lon={meta_limon['lon']:.4f}")

formas = cargar_formas_conocidas(usar_residuo=True)
ff, sj = formas["finca_favorita"], formas["san_jose"]

dist_limon_ff = _haversine_km(meta_limon["lat"], meta_limon["lon"], ff["lat"], ff["lon"])
dist_sj_ff = _haversine_km(sj["lat"], sj["lon"], ff["lat"], ff["lon"])
print(f"\nDistancia Limón <-> Finca Favorita: {dist_limon_ff:.1f} km")
print(f"Distancia San José <-> Finca Favorita (donante actual): {dist_sj_ff:.1f} km")
print(f"Limón está {dist_sj_ff / dist_limon_ff:.1f}x más cerca de Finca Favorita que San José.")

hm_limon = heatmap_json_desde_epw(df_limon)
ws_limon = excedencia_json_desde_epw_residual(df_limon, hm_limon)
formas_con_limon = dict(formas)
formas_con_limon["limon"] = dict(nombre="Limón Intl. A.P. (Caribe)", lat=meta_limon["lat"],
                                  lon=meta_limon["lon"], elevacion_m=meta_limon["elevacion_m"],
                                  ws_json=ws_limon, hm_json=hm_limon, df_real=df_limon)

donante, dist = vecino_mas_cercano(ff["lat"], ff["lon"], formas_con_limon, excluir="finca_favorita")
print(f"\nCon Limón disponible, vecino_mas_cercano() elige: {formas_con_limon[donante]['nombre']} ({dist:.1f} km)")

Limón: media=2.152 m/s, elev=2.1m, lat=9.9580, lon=-83.0220



Distancia Limón <-> Finca Favorita: 63.8 km
Distancia San José <-> Finca Favorita (donante actual): 178.6 km
Limón está 2.8x más cerca de Finca Favorita que San José.

Con Limón disponible, vecino_mas_cercano() elige: Limón Intl. A.P. (Caribe) (63.8 km)


In [6]:
# Prueba "vecino + verdad conocida" (mismo método que Hallazgo 22): tomar prestada la FORMA real
# de cada candidato, escalada a la media REAL YA CONOCIDA de Finca Favorita -- aísla la pregunta de
# forma de la de magnitud.
media_real_ff = float(ff["df_real"]["WS10M"].mean())
r_real = simular(ff["df_real"], 3.0, "medium_tulip", 3, elevacion_m=ff["elevacion_m"])
print(f"Finca Favorita -- media real: {media_real_ff:.3f} m/s, producción real: {r_real['kwh_anual']:.3f} kWh/año\n")

for nombre_donante, clave_donante in (("San José (donante actual)", "san_jose"), ("Limón (candidato nuevo)", "limon")):
    donante_dict = formas_con_limon[clave_donante]
    df_prestado, _ = generar_clima_gwa(donante_dict["ws_json"], donante_dict["hm_json"],
                                        media_objetivo=media_real_ff)
    r_prestado = simular(df_prestado, 3.0, "medium_tulip", 3, elevacion_m=ff["elevacion_m"])
    error_pct = (r_prestado["kwh_anual"] / r_real["kwh_anual"] - 1) * 100
    print(f"  Donante = {nombre_donante:28s} -> kWh={r_prestado['kwh_anual']:8.3f}  error={error_pct:+8.1f}%")

Finca Favorita -- media real: 1.413 m/s, producción real: 7.439 kWh/año

  Donante = San José (donante actual)    -> kWh=   8.864  error=   +19.2%
  Donante = Limón (candidato nuevo)      -> kWh=  12.852  error=   +72.8%


**Resultado real, no el esperado:** aunque Limón está 2.8x más cerca (63.8km vs 178.6km) y
`vecino_mas_cercano()` lo elegiría automáticamente, su FORMA real predice PEOR la producción de
Finca Favorita que la de San José (+72.8% de error contra +19.2%) -- empeora, no mejora.

In [7]:
# Por qué, verificado -- comparar la FORMA (no la magnitud) via E[v^3]/media^3 (EPF) y horas de calma.
import numpy as np

def epf(ws):
    ws = np.asarray(ws, dtype=float)
    return float(np.mean(ws ** 3) / np.mean(ws) ** 3)

def frac_calma(ws, umbral=1.0):
    ws = np.asarray(ws, dtype=float)
    return float(np.mean(ws < umbral) * 100)

vals_sj = np.array([r["val"] for r in sj["ws_json"]])
print(f"{'sitio':30s} {'media_m_s':>10s} {'EPF':>8s} {'%horas<1.0m/s':>14s}")
print(f"{'Finca Favorita (real)':30s} {media_real_ff:10.3f} {epf(ff['df_real']['WS10M'].values):8.3f} "
      f"{frac_calma(ff['df_real']['WS10M'].values):13.1f}%")
print(f"{'Limón (real)':30s} {df_limon['WS10M'].mean():10.3f} {epf(df_limon['WS10M'].values):8.3f} "
      f"{frac_calma(df_limon['WS10M'].values):13.1f}%")
print(f"{'San José (curva GWA, aprox.)':30s} {np.mean(vals_sj):10.3f} {epf(vals_sj):8.3f} {'N/A (curva)':>13s}")

sitio                           media_m_s      EPF  %horas<1.0m/s
Finca Favorita (real)               1.413    1.617          25.2%
Limón (real)                        2.152    2.515          15.7%
San José (curva GWA, aprox.)        3.669    1.077   N/A (curva)


**Diagnóstico:** Finca Favorita tiene 25.2% de horas por debajo de 1.0 m/s (bastante calma/protegida
-- consistente con la hipótesis ya documentada en Hallazgo 26 de terreno costero-boscoso). Limón, un
aeropuerto abierto directo sobre la costa, tiene solo 15.7% -- más ventoso y más variable (EPF 2.515
contra 1.617 de Finca Favorita, +55.6%). San José, pese a estar mucho más lejos y en un régimen
climático distinto, tiene un EPF más parecido (1.077, -33.4% contra el real) -- una forma más
"amortiguada", más parecida a la de un sitio protegido como Finca Favorita que la de un aeropuerto
costero expuesto como Limón.

**Conclusión honesta:** la cercanía geográfica (e incluso compartir el mismo tramo de costa) no
garantiza una forma parecida -- la EXPOSICIÓN local (aeropuerto abierto vs. finca protegida/con
cobertura vegetal) pesa más que la distancia acá. Esto es exactamente el tipo de señal que el
downscaling topográfico del informe de Pablo (TPI, rugosidad vía WorldCover) apunta a capturar --
confirma que el problema es real, no que ya esté resuelto con Köppen o con más estaciones cercanas
solamente.

San José sigue siendo el mejor donante real disponible para Finca Favorita entre los sitios
conocidos hoy. Las otras 7 estaciones descargadas (Chacarita-Puntarenas, Palmar Sur, Parrita, Paso
Canoas, Puntarenas, San José-Bolaños, San José-La Sabana) están guardadas en `datos_clima/epw_real/`
pero sin analizar todavía -- quedan como candidatas si se decide seguir buscando, o como enriquecimiento
general del catálogo de sitios conocidos más allá de este problema específico.

**Vale la pena notar:** Finca Favorita produce apenas 7.4 kWh/año real -- un sitio de recurso eólico
muy pobre en términos absolutos (1.413 m/s de media). Cualquier recomendación real para un cliente
ahí ya diría "recurso insuficiente" independientemente del error porcentual exacto -- vale la pena
que Pablo decida si perseguir más precisión en este sitio específico es la mejor inversión de tiempo
ahora mismo.